# Gemma-12B: Neutral-topic follow-up pilot

Follow-up to the climate generalization pilot (`experiments/e1_climate/gemma4-12b/`), which found
Gemma-12B *inverts* on that pilot: it prefers the LOWER-engagement post in all 21 of 21 tested
off-diagonal scale-pairs, the opposite of every other model's normal conformity direction on the
same pilot, and the opposite of Gemma-12B's own behavior on the main Pop-vs-Latin study. Diagonal
(tied-engagement) accuracy there is 74.3%, and refusals are near-zero (0.2%), so the inversion is
real, not a parsing artifact -- confirmed 2026-08-28 following the supervisor's request to triple-check it.

This pilot asks whether that inversion is specific to the climate/renewable-energy content, or a
general property of this model's behavior on this chart-plus-engagement format regardless of topic.
Same protocols as the climate pilot (baseline single-image like/scroll + logprobs, single-image
across the 6 `metrics/realistic` engagement scales + logprobs, full 7x7 paired A/B `metrics` grid),
same underlying fabricated data (byte-identical numeric series, verified against the climate pilot's
own generator for all 25 posts), only the topic swapped to a maximally generic, content-free frame
("Product A" vs. "Product B" quarterly revenue) -- pointed at `neutral_pilot/posts/`.

**25 posts (not 50/100): this is a scoped pilot, not a full replication. Gemma-12B only, per the
supervisor's request -- not rolled out to the other 5 models.**

In [1]:
import sys, subprocess

# 1. Uninstall torchaudio
subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

# 2. Install PyTorch with CUDA 12.4
subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

# 3. Install latest transformers and accelerate (allowing pip to pull compatible tokenizers naturally)
subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

print("✅ Installation complete — restart the kernel now")



✅ Installation complete — restart the kernel now


Restart kernel after running the setup cell above.

In [2]:
!nvidia-smi

Fri Aug 28 15:05:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:CF:00.0 Off |                   On |
| N/A   26C    P0             76W /  700W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

In [3]:
# --- HF Auth ---
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

# --- Path setup ---
from pathlib import Path
ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

# --- Load Gemma model ---
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/gemma-4-12B-it"
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")
device = model.device

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

In [4]:
from e1_utils.sampling import build_paired_sample
from e1_utils.e1_optimized import (
    LIKE_PROMPT_SINGLE, LIKE_PROMPT_YESNO, LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_baseline, run_e1_metrics, run_e1_metrics_paired
)
from e1_utils.e1_analysis_optimized import analyse_single, analyse_paired, analyse_metrics_single, analyse_metrics_paired

# --- Configuration: neutral-topic follow-up pilot, NOT the main benchmarking/ pool ---
EXPERIMENT_DIR = Path().resolve().parent        # experiments/e1_neutral/  -- Gemma-12B only for this pilot
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42
SAMPLE_SIZE = 25

correct_dir = ROOT_DIR / "neutral_pilot/posts/correct/PNGs"
incorrect_dir = ROOT_DIR / "neutral_pilot/posts/incorrect/PNGs"
all_images = build_paired_sample(correct_dir, incorrect_dir, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

# metrics/realistic condition only -- the condition that showed the strongest conformity effect
# in the main study (Section 6.2), and the one the climate-pilot inversion was found under
correct_base = ROOT_DIR / "neutral_pilot/posts/correct/PNGs/metrics/realistic"
incorrect_base = ROOT_DIR / "neutral_pilot/posts/incorrect/PNGs/metrics/realistic"


Found 25 images in /home/jovyan/conformity-llms-facebook-posts/neutral_pilot/posts/correct/PNGs
✅ Saved selection to /home/jovyan/conformity-llms-facebook-posts/experiments/e1_neutral/selected_images.json
✅ All selected numbers verified in both correct and incorrect folders.
Selected 25 pairs → 50 images total


In [5]:
from e1_utils.inference_gemma import run_inference_gemma

In [6]:
from e1_utils.e1_optimized import (
    run_e1_baseline_logprobs, run_e1_metrics_logprobs,
    LIKE_CANDIDATES_SINGLE, LIKE_CANDIDATES_YESNO
)

from e1_utils.inference_gemma import run_inference_with_scores_gemma

## Approach 1 -- single image, like/scroll, baseline (0 engagement)

In [7]:
run_e1_baseline(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_baseline.json",
              inference_fn=run_inference_gemma)

✅ 001_correct → scroll
✅ 001_incorrect → scroll
✅ 002_correct → scroll
✅ 002_incorrect → scroll
✅ 003_correct → scroll
✅ 003_incorrect → scroll
✅ 004_correct → scroll
✅ 004_incorrect → scroll
✅ 005_correct → scroll
✅ 005_incorrect → scroll
✅ 006_correct → scroll
✅ 006_incorrect → scroll
✅ 007_correct → scroll
✅ 007_incorrect → scroll
✅ 008_correct → scroll
✅ 008_incorrect → scroll
✅ 009_correct → scroll
✅ 009_incorrect → scroll
✅ 010_correct → scroll
✅ 010_incorrect → scroll
✅ 011_correct → scroll
✅ 011_incorrect → scroll
✅ 012_correct → scroll
✅ 012_incorrect → scroll
✅ 013_correct → scroll
✅ 013_incorrect → scroll
✅ 014_correct → scroll
✅ 014_incorrect → scroll
✅ 015_correct → scroll
✅ 015_incorrect → scroll
✅ 016_correct → scroll
✅ 016_incorrect → scroll
✅ 017_correct → scroll
✅ 017_incorrect → scroll
✅ 018_correct → scroll
✅ 018_incorrect → scroll
✅ 019_correct → scroll
✅ 019_incorrect → scroll
✅ 020_correct → scroll
✅ 020_incorrect → scroll
✅ 021_correct → scroll
✅ 021_incorrect →

In [8]:
run_e1_baseline_logprobs(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_baseline_logprobs.json", score_fn=run_inference_with_scores_gemma)

✅ 001_correct → scroll {'like': {'logprob': -24.42194366455078, 'prob_forced_choice': 0.0009253848866495988}, 'scroll': {'logprob': -17.43756866455078, 'prob_forced_choice': 0.9990746151133504}}
✅ 001_incorrect → scroll {'like': {'logprob': -26.781253814697266, 'prob_forced_choice': 0.00039977102951818487}, 'scroll': {'logprob': -18.957035064697266, 'prob_forced_choice': 0.9996002289704817}}
✅ 002_correct → scroll {'like': {'logprob': -25.234434127807617, 'prob_forced_choice': 0.0004108485376346404}, 'scroll': {'logprob': -17.437559127807617, 'prob_forced_choice': 0.9995891514623653}}
✅ 002_incorrect → scroll {'like': {'logprob': -25.968761444091797, 'prob_forced_choice': 0.0004955359475635505}, 'scroll': {'logprob': -18.359386444091797, 'prob_forced_choice': 0.9995044640524365}}
✅ 003_correct → scroll {'like': {'logprob': -24.140853881835938, 'prob_forced_choice': 0.0008693673928062171}, 'scroll': {'logprob': -17.093978881835938, 'prob_forced_choice': 0.9991306326071938}}
✅ 003_incorr

In [9]:
analyse_single(OUTPUT_DIR, "e1_results_baseline.json", like_answer="like")


Single image analysis: e1_results_baseline
=== Summary ===


,metric,value
0,overall_like_rate_%,0.0
1,like_rate_correct_%,0.0
2,like_rate_incorrect_%,0.0


=== Per Image Results ===


,image,variant,prompt,answer
0,001_correct,correct,You are shown a social media post.\nYou can ei...,scroll
1,001_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
2,002_correct,correct,You are shown a social media post.\nYou can ei...,scroll
3,002_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
4,003_correct,correct,You are shown a social media post.\nYou can ei...,scroll
5,003_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
6,004_correct,correct,You are shown a social media post.\nYou can ei...,scroll
7,004_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll
8,005_correct,correct,You are shown a social media post.\nYou can ei...,scroll
9,005_incorrect,incorrect,You are shown a social media post.\nYou can ei...,scroll


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_neutral/gemma4-12b/outputs/e1_analysis_baseline.csv


## Approach 1 variant -- single image, like/scroll, across the 6 `metrics/realistic` engagement scales

In [10]:
run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_metrics.json",
              inference_fn=run_inference_gemma)

✅ 001_correct_10 → scroll
✅ 001_incorrect_10 → scroll
✅ 002_correct_10 → scroll
✅ 002_incorrect_10 → scroll
✅ 003_correct_10 → scroll
✅ 003_incorrect_10 → scroll
✅ 004_correct_10 → scroll
✅ 004_incorrect_10 → scroll
✅ 005_correct_10 → scroll
✅ 005_incorrect_10 → scroll
✅ 006_correct_10 → scroll
✅ 006_incorrect_10 → scroll
✅ 007_correct_10 → scroll
✅ 007_incorrect_10 → scroll
✅ 008_correct_10 → scroll
✅ 008_incorrect_10 → scroll
✅ 009_correct_10 → scroll
✅ 009_incorrect_10 → scroll
✅ 010_correct_10 → scroll
✅ 010_incorrect_10 → scroll
✅ 011_correct_10 → scroll
✅ 011_incorrect_10 → scroll
✅ 012_correct_10 → scroll
✅ 012_incorrect_10 → scroll
✅ 013_correct_10 → scroll
✅ 013_incorrect_10 → scroll
✅ 014_correct_10 → scroll
✅ 014_incorrect_10 → scroll
✅ 015_correct_10 → scroll
✅ 015_incorrect_10 → scroll
✅ 016_correct_10 → scroll
✅ 016_incorrect_10 → scroll
✅ 017_correct_10 → scroll
✅ 017_incorrect_10 → scroll
✅ 018_correct_10 → scroll
✅ 018_incorrect_10 → scroll
✅ 019_correct_10 → scroll
✅ 

In [11]:
run_e1_metrics_logprobs(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_metrics_logprobs.json", score_fn=run_inference_with_scores_gemma)

✅ 001_correct_10 → scroll {'like': {'logprob': -25.062528610229492, 'prob_forced_choice': 0.0008969393236719385}, 'scroll': {'logprob': -18.046903610229492, 'prob_forced_choice': 0.9991030606763281}}
✅ 001_incorrect_10 → scroll {'like': {'logprob': -26.625001907348633, 'prob_forced_choice': 0.0004807662756410833}, 'scroll': {'logprob': -18.985353469848633, 'prob_forced_choice': 0.9995192337243589}}
✅ 002_correct_10 → scroll {'like': {'logprob': -24.781335830688477, 'prob_forced_choice': 0.0009110511944006454}, 'scroll': {'logprob': -17.781335830688477, 'prob_forced_choice': 0.9990889488055994}}
✅ 002_incorrect_10 → scroll {'like': {'logprob': -25.562517166137695, 'prob_forced_choice': 0.0007554056327295276}, 'scroll': {'logprob': -18.375017166137695, 'prob_forced_choice': 0.9992445943672705}}
✅ 003_correct_10 → scroll {'like': {'logprob': -23.25232696533203, 'prob_forced_choice': 0.002545947704947981}, 'scroll': {'logprob': -17.28162384033203, 'prob_forced_choice': 0.997454052295052}}


In [12]:
analyse_metrics_single(OUTPUT_DIR, "e1_results_metrics.json", like_answer="like")


Metrics single image analysis: e1_results_metrics
=== Overall Summary ===


,metric,value
0,overall_like_rate_%,0.0
1,like_rate_correct_%,0.0
2,like_rate_incorrect_%,0.0


=== Rate per Scale Value ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:116: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_scale = df.groupby("scale_value").apply(lambda g: pd.Series({


,scale_value,total_images,overall_like_rate_%,like_rate_correct_%,like_rate_incorrect_%
0,10,50.0,0.0,0.0,0.0
1,100,50.0,0.0,0.0,0.0
2,1000,50.0,0.0,0.0,0.0
3,10000,50.0,0.0,0.0,0.0
4,100000,50.0,0.0,0.0,0.0
5,1000000,50.0,0.0,0.0,0.0


=== Per Image Results ===


,image,num,variant,scale_value,prompt,answer
0,001_correct_10,001,correct,10,You are shown a social media post.\nYou can ei...,scroll
2,002_correct_10,002,correct,10,You are shown a social media post.\nYou can ei...,scroll
4,003_correct_10,003,correct,10,You are shown a social media post.\nYou can ei...,scroll
6,004_correct_10,004,correct,10,You are shown a social media post.\nYou can ei...,scroll
8,005_correct_10,005,correct,10,You are shown a social media post.\nYou can ei...,scroll
...,...,...,...,...,...,...
291,021_incorrect_1000000,021,incorrect,1000000,You are shown a social media post.\nYou can ei...,scroll
293,022_incorrect_1000000,022,incorrect,1000000,You are shown a social media post.\nYou can ei...,scroll
295,023_incorrect_1000000,023,incorrect,1000000,You are shown a social media post.\nYou can ei...,scroll
297,024_incorrect_1000000,024,incorrect,1000000,You are shown a social media post.\nYou can ei...,scroll


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_neutral/gemma4-12b/outputs/e1_analysis_metrics.csv


## Approach 2 -- paired A/B forced choice, full 7x7 `metrics/realistic` disparity grid

In [13]:
import time
start = time.time()
run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
              prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_metrics_paired.json",
              inference_fn=run_inference_gemma)
elapsed = time.time() - start
print(f"\n⏱ Total runtime: {elapsed/60:.1f} min")

✅ 001_correct0_vs_incorrect0 → liked correct (answered A)
✅ 002_correct0_vs_incorrect0 → liked correct (answered A)
✅ 003_correct0_vs_incorrect0 → liked correct (answered A)
✅ 004_correct0_vs_incorrect0 → liked correct (answered B)
✅ 005_correct0_vs_incorrect0 → liked correct (answered A)
✅ 006_correct0_vs_incorrect0 → liked correct (answered B)
✅ 007_correct0_vs_incorrect0 → liked correct (answered A)
✅ 008_correct0_vs_incorrect0 → liked correct (answered A)
✅ 009_correct0_vs_incorrect0 → liked correct (answered A)
✅ 010_correct0_vs_incorrect0 → liked correct (answered B)
✅ 011_correct0_vs_incorrect0 → liked incorrect (answered A)
✅ 012_correct0_vs_incorrect0 → liked correct (answered B)
✅ 013_correct0_vs_incorrect0 → liked correct (answered A)
✅ 014_correct0_vs_incorrect0 → liked incorrect (answered A)
✅ 015_correct0_vs_incorrect0 → liked correct (answered A)
✅ 016_correct0_vs_incorrect0 → liked incorrect (answered A)
✅ 017_correct0_vs_incorrect0 → liked correct (answered A)
✅ 018_co

In [14]:
analyse_metrics_paired(OUTPUT_DIR, "e1_results_metrics_paired.json")


Metrics paired A/B analysis: e1_results_metrics_paired
=== Overall Summary ===


,metric,value
0,overall_liked_correct_%,59.67
1,overall_liked_incorrect_%,40.24
2,invalid_answer_%,0.08


=== Liked Correct Rate per Scale Pair ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:163: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_pair = df.groupby("pair").apply(lambda g: pd.Series({


,pair,total_pairs,liked_correct_%,liked_incorrect_%,invalid_%
0,0_vs_0,25.0,84.0,12.0,4.0
1,0_vs_10,25.0,88.0,12.0,0.0
2,0_vs_100,25.0,60.0,40.0,0.0
3,0_vs_1000,25.0,72.0,28.0,0.0
4,0_vs_10000,25.0,88.0,12.0,0.0
5,0_vs_100000,25.0,60.0,40.0,0.0
6,0_vs_1000000,25.0,76.0,24.0,0.0
7,10_vs_0,25.0,56.0,44.0,0.0
8,10_vs_10,25.0,72.0,28.0,0.0
9,10_vs_100,25.0,72.0,28.0,0.0


=== Per Pair Results ===


,image,num,correct_scale,incorrect_scale,post_a_variant,post_b_variant,prompt,answer,liked_variant,pair
0,001_correct0_vs_incorrect0,001,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
1,002_correct0_vs_incorrect0,002,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
2,003_correct0_vs_incorrect0,003,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
3,004_correct0_vs_incorrect0,004,0,0,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct,0_vs_0
4,005_correct0_vs_incorrect0,005,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
...,...,...,...,...,...,...,...,...,...,...
1220,021_correct1000000_vs_incorrect1000000,021,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",A,incorrect,1000000_vs_1000000
1221,022_correct1000000_vs_incorrect1000000,022,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",A,incorrect,1000000_vs_1000000
1222,023_correct1000000_vs_incorrect1000000,023,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",A,incorrect,1000000_vs_1000000
1223,024_correct1000000_vs_incorrect1000000,024,1000000,1000000,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,1000000_vs_1000000


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_neutral/gemma4-12b/outputs/e1_analysis_metrics_paired.csv
